# Thu thập dữ liệu Art Institute of Chicago cho CDNMF

Notebook gồm hai bước: crawl tác phẩm từ API Art Institute of Chicago, sau đó cân bằng nhãn và xuất dữ liệu CDNMF. Mỗi node là một tác phẩm (ID `id`); các tác phẩm cùng `artist_id` được nối với tối đa 20 láng giềng. Nhãn dùng để đánh giá là `department_title`; `title` và `artist_title` chỉ lưu làm thông tin, không dùng làm feature. CSV crawl vẫn được lưu tại `Database/artic_25000`; bộ đã xử lý được lưu riêng tại `Database/artic_department_25000` để giữ nguyên kết quả nhãn cũ.

Chạy cell crawl trước. Nếu quá trình bị ngắt, chạy lại cell đó để tiếp tục từ checkpoint.

In [ ]:
from pathlib import Path
from crawl_artic_cdnmf import crawl_artworks

ROOT = Path.cwd()
RAW_DIR = ROOT / "Database" / "artic_25000"
OUTPUT_DIR = ROOT / "Database" / "artic_department_25000"
RAW_CSV = RAW_DIR / "artworks_raw.csv"
CRAWL_LIMIT = 25_000

if not (ROOT / "crawl_artic_cdnmf.py").is_file():
    raise FileNotFoundError("Hãy mở notebook từ thư mục gốc repository CDNMF.")

collected_count = crawl_artworks(CRAWL_LIMIT, RAW_CSV, label_field="department_title")
print(f"Đã lưu {collected_count:,} artwork có ID, tác giả, tiêu đề và department_title vào {RAW_CSV}")
print(f"Checkpoint: {RAW_CSV.with_suffix('.checkpoint.json')}")

## 2. Tiền xử lý CSV

Dữ liệu sau xử lý giữ tối đa 21.000 artwork. Chọn 6–8 `department_title` có ít nhất 100 mẫu; cân bằng sao cho lớp lớn nhất không vượt quá 3 lần lớp nhỏ nhất. Mỗi tác giả có tối đa 200 tác phẩm và còn ít nhất 11 tác phẩm sau lọc. Các bước lọc được lặp lại để duy trì ngưỡng này.

Feature mặc định gồm `material_titles`, `technique_titles`, `place_of_origin`, `medium_display`; thêm trường có độ phủ cao hơn giữa `subject_titles`/`style_titles` nếu đạt ít nhất 50%. `title`, `artist_title`, `department_title` và `artwork_type_title` không nằm trong feature. Graph nối các tác phẩm cùng tác giả, tối đa 20 láng giềng mỗi node; số cộng đồng k bằng số lớp còn lại.

In [ ]:
from crawl_artic_cdnmf import prepare_dataset, read_artwork_csv

PROCESSED_LIMIT = 21_000
FEATURE_LIMIT = 128
NEIGHBORS = 20
MIN_CLASSES = 6
MAX_CLASSES = 8
MIN_CLASS_SAMPLES = 100
MAX_CLASS_RATIO = 3.0
MAX_PER_ARTIST = 200
MIN_PER_ARTIST = 11
OPTIONAL_FEATURE_COVERAGE = 0.5

if not RAW_CSV.is_file():
    raise FileNotFoundError("Chạy cell thu thập dữ liệu trước để tạo artworks_raw.csv.")

result = prepare_dataset(
    records=read_artwork_csv(RAW_CSV, CRAWL_LIMIT),
    output_dir=OUTPUT_DIR,
    processed_limit=PROCESSED_LIMIT,
    label_field="department_title",
    feature_limit=FEATURE_LIMIT,
    neighbors=NEIGHBORS,
    min_classes=MIN_CLASSES,
    max_classes=MAX_CLASSES,
    min_class_samples=MIN_CLASS_SAMPLES,
    max_class_ratio=MAX_CLASS_RATIO,
    max_per_artist=MAX_PER_ARTIST,
    min_per_artist=MIN_PER_ARTIST,
    optional_feature_coverage=OPTIONAL_FEATURE_COVERAGE,
)
print(result)
print(f"CSV gốc: {RAW_CSV}")
print(f"CSV sau tiền xử lý: {OUTPUT_DIR / 'artworks.csv'}")